# Academic Factor Data Dataset

Fama-French and AQR factor returns for benchmarking and risk adjustment.

| Property | Value |
|----------|-------|
| **Provider** | Ken French Library, AQR |
| **Asset Class** | Factor Returns |
| **Frequency** | Monthly (daily available) |
| **Factors** | FF3, FF5, Momentum, QMJ, BAB |
| **Coverage** | 1926-present (FF), varies (AQR) |
| **Size** | ~5 MB |
| **API Key** | None (free) |
| **Loader** | `load_ff_factors()`, `load_aqr_factors()` |

In [1]:
"""Academic Factor Data - download, explore, and update workflow."""

import json
from pathlib import Path

import polars as pl

## 1. Configuration

Academic factor data is **provider-defined** (no local config file). Each provider
maintains their own factor definitions and data format.

In [2]:
print("=== Academic Factor Configuration ===")
print("\nFama-French (Ken French Library):")
print("  - FF3: Mkt-RF, SMB, HML")
print("  - FF5: FF3 + RMW, CMA")
print("  - Momentum: MOM")
print("  - Coverage: 1926-present")
print("\nAQR Research:")
print("  - QMJ: Quality Minus Junk")
print("  - BAB: Betting Against Beta")
print("  - VME: Value Minus Everything")
print("  - HML Devil: Industry-adjusted value")
print("  - Coverage: varies by factor")

=== Academic Factor Configuration ===

Fama-French (Ken French Library):
  - FF3: Mkt-RF, SMB, HML
  - FF5: FF3 + RMW, CMA
  - Momentum: MOM
  - Coverage: 1926-present

AQR Research:
  - QMJ: Quality Minus Junk
  - BAB: Betting Against Beta
  - VME: Value Minus Everything
  - HML Devil: Industry-adjusted value
  - Coverage: varies by factor


## 2. API Key Setup

**No API key required.** Both Ken French Library and AQR provide free public access.

In [3]:
print("Ken French Library: Free, no API key required")
print("  URL: https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html")
print("\nAQR Research: Free, no API key required")
print("  URL: https://www.aqr.com/Insights/Datasets")

Ken French Library: Free, no API key required
  URL: https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html

AQR Research: Free, no API key required
  URL: https://www.aqr.com/Insights/Datasets


## 3. Download Data

The `ml4t-data` library handles downloading and caching factor data.

In [4]:
def download_ff_factors(
    datasets: list[str] | None = None, frequency: str = "monthly", dry_run: bool = False
):
    """Download Fama-French factor data.

    Args:
        datasets: Specific datasets to download (default: core factors)
        frequency: "monthly" or "daily"
        dry_run: If True, show what would be downloaded
    """
    from ml4t.data.providers.fama_french import FamaFrenchProvider

    from utils import ML4T_DATA_PATH

    output_dir = ML4T_DATA_PATH / "factors" / "fama-french"

    # Default core datasets
    if datasets is None:
        datasets = ["ff3", "ff5", "mom"]

    print("=== Fama-French Download ===")
    print(f"Datasets: {datasets}")
    print(f"Frequency: {frequency}")
    print(f"Output: {output_dir}")

    if dry_run:
        print("\n[DRY RUN] Would download:")
        for ds in datasets:
            print(f"  - {ds}")
        return

    output_dir.mkdir(parents=True, exist_ok=True)
    provider = FamaFrenchProvider(cache_path=output_dir, use_cache=True)

    print(f"\nDownloading {len(datasets)} datasets...")
    for dataset in datasets:
        print(f"  {dataset}...", end=" ", flush=True)
        try:
            df = provider.fetch(dataset, frequency=frequency)
            print(f"OK ({len(df):,} rows)")
        except Exception as e:
            print(f"ERROR: {e}")

    print("\n=== Complete ===")
    print(f"Data saved to: {output_dir}")


def download_aqr_factors(datasets: list[str] | None = None, dry_run: bool = False):
    """Download AQR factor data.

    Args:
        datasets: Specific datasets to download (default: core factors)
        dry_run: If True, show what would be downloaded
    """
    from ml4t.data.providers.aqr import AQRProvider

    from utils import ML4T_DATA_PATH

    output_dir = ML4T_DATA_PATH / "factors" / "aqr"

    # Default core datasets
    if datasets is None:
        datasets = ["qmj", "bab"]

    print("=== AQR Download ===")
    print(f"Datasets: {datasets}")
    print(f"Output: {output_dir}")

    if dry_run:
        print("\n[DRY RUN] Would download:")
        for ds in datasets:
            print(f"  - {ds}")
        return

    output_dir.mkdir(parents=True, exist_ok=True)
    provider = AQRProvider(cache_path=output_dir)

    print(f"\nDownloading {len(datasets)} datasets...")
    for dataset in datasets:
        print(f"  {dataset}...", end=" ", flush=True)
        try:
            df = provider.fetch(dataset)
            print(f"OK ({len(df):,} rows)")
        except Exception as e:
            print(f"ERROR: {e}")

    print("\n=== Complete ===")
    print(f"Data saved to: {output_dir}")

### Download Fama-French Factors

In [5]:
# Uncomment to download
# download_ff_factors()

### Download AQR Factors

In [6]:
# Uncomment to download
# download_aqr_factors()

### Dry Run (Preview)

In [7]:
download_ff_factors(dry_run=True)

=== Fama-French Download ===
Datasets: ['ff3', 'ff5', 'mom']
Frequency: monthly
Output: data/factors/fama-french

[DRY RUN] Would download:
  - ff3
  - ff5
  - mom


## 4. Load and Explore

Once downloaded, use the loaders throughout the book:

In [8]:
from data import load_aqr_factors, load_ff_factors

### Fama-French Factors

In [9]:
# Load Fama-French factors
ff = load_ff_factors()

print(f"Shape: {ff.shape}")
print(f"Columns: {ff.columns}")
print(f"Date range: {ff['timestamp'].min()} to {ff['timestamp'].max()}")
print(f"Memory: {ff.estimated_size('mb'):.1f} MB")

Shape: (752, 7)
Columns: ['timestamp', 'Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'RF']
Date range: 1963-07-01 to 2026-02-01
Memory: 0.0 MB


In [10]:
# Preview
ff.tail(10)

timestamp,Mkt-RF,SMB,HML,RMW,CMA,RF
date,f64,f64,f64,f64,f64,f64
2025-05-01,0.0606,-0.0072,-0.0288,0.0129,0.0251,0.0038
2025-06-01,0.0486,-0.0002,-0.0161,-0.032,0.0144,0.0034
2025-07-01,0.0198,-0.0014,-0.0127,-0.0028,-0.0207,0.0034
2025-08-01,0.0185,0.0488,0.0442,-0.0069,0.0208,0.0038
2025-09-01,0.0339,-0.0218,-0.0105,-0.0205,-0.0222,0.0033
2025-10-01,0.0196,-0.0131,-0.031,-0.0524,-0.0403,0.0037
2025-11-01,-0.0013,0.0147,0.0376,0.0144,0.0068,0.003
2025-12-01,-0.0036,-0.0022,0.0242,0.004,0.0037,0.0034
2026-01-01,0.0103,0.0326,0.0372,0.0182,0.0183,0.003


In [11]:
# Factor statistics (annualized)
factor_cols = [c for c in ff.columns if c not in ["timestamp", "date"]]
print("Factor Annualized Statistics (%):")
for col in factor_cols[:6]:
    series = ff[col].drop_nulls()
    mean_annual = series.mean() * 12  # Monthly to annual
    vol_annual = series.std() * (12**0.5)
    sharpe = mean_annual / vol_annual if vol_annual > 0 else 0
    print(f"  {col:8s}: mean={mean_annual:6.2f}, vol={vol_annual:6.2f}, SR={sharpe:.2f}")

Factor Annualized Statistics (%):
  Mkt-RF  : mean=  0.07, vol=  0.15, SR=0.46
  SMB     : mean=  0.02, vol=  0.10, SR=0.21
  HML     : mean=  0.04, vol=  0.10, SR=0.34
  RMW     : mean=  0.03, vol=  0.08, SR=0.42
  CMA     : mean=  0.03, vol=  0.07, SR=0.42
  RF      : mean=  0.04, vol=  0.01, SR=4.82


### AQR Factors

In [12]:
# Load AQR factors
aqr = load_aqr_factors()

print(f"Shape: {aqr.shape}")
print(f"Columns: {aqr.columns}")
print(f"Date range: {aqr['timestamp'].min()} to {aqr['timestamp'].max()}")

Shape: (822, 30)
Columns: ['timestamp', 'AUS', 'AUT', 'BEL', 'CAN', 'CHE', 'DEU', 'DNK', 'ESP', 'FIN', 'FRA', 'GBR', 'GRC', 'HKG', 'IRL', 'ISR', 'ITA', 'JPN', 'NLD', 'NOR', 'NZL', 'PRT', 'SGP', 'SWE', 'USA', 'Global', 'Global Ex USA', 'Europe', 'North America', 'Pacific']
Date range: 1957-07-01 00:00:00 to 2025-12-01 00:00:00


In [13]:
# Preview
aqr.tail(10)

timestamp,AUS,AUT,BEL,CAN,CHE,DEU,DNK,ESP,FIN,FRA,GBR,GRC,HKG,IRL,ISR,ITA,JPN,NLD,NOR,NZL,PRT,SGP,SWE,USA,Global,Global Ex USA,Europe,North America,Pacific
datetime[ns],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2025-03-01 00:00:00,-0.007828,0.029519,-0.042761,0.021093,0.042585,0.018169,-0.087983,-0.010841,-0.013099,-0.052091,0.016994,-0.0104,0.028615,0.012799,0.031601,-0.085393,-0.019749,-0.003039,0.062309,0.016388,0.025513,0.00923,0.036167,0.035278,0.022699,-0.000942,-0.004567,0.03461,-0.001338
2025-04-01 00:00:00,0.007435,-0.046474,0.01118,-0.023238,0.01169,0.033374,-0.068685,0.018211,-0.011406,-0.01625,0.001274,0.052583,-0.016407,0.066556,-0.021292,-0.044949,0.014708,-0.022464,0.007533,0.01262,-0.042121,0.00042,-0.035669,-0.020094,-0.013819,-0.00267,-0.003262,-0.020248,0.003022
2025-05-01 00:00:00,-0.040166,0.008512,-0.033412,0.009043,-0.018815,-0.012478,0.042664,-0.014984,-0.008306,-0.010874,-0.038765,-0.069848,-0.001297,-0.053195,0.020805,-0.034816,-0.008749,0.041948,0.001849,0.053586,-0.014323,0.025185,-0.002881,-0.017466,-0.014582,-0.009652,-0.013751,-0.016104,-0.008591
2025-06-01 00:00:00,-0.003248,0.007846,0.046128,-0.005309,-0.020654,-0.003016,0.02235,-0.021876,0.040276,0.004203,-0.060986,-0.028797,-0.039664,-0.104203,-0.062785,-0.000602,0.013707,0.02255,-0.006647,-0.00163,0.003169,-0.000282,-0.013404,-0.042963,-0.030942,-0.009992,-0.01392,-0.041039,-0.005798
2025-07-01 00:00:00,-0.012351,0.023015,-0.000526,-0.015691,-0.022335,0.004695,-0.107453,-0.031385,-0.029448,0.002948,-0.00161,-0.076286,-0.03803,0.02113,-0.013823,-0.054555,-0.024847,-0.065462,0.01833,-0.000879,-0.029839,0.003063,0.004894,-0.008207,-0.011998,-0.018777,-0.013978,-0.008583,-0.026022
2025-08-01 00:00:00,-0.045507,-0.022785,0.012979,-0.033649,0.019429,0.011466,0.091161,-0.010314,-0.064168,-0.006143,-0.025266,-0.015942,-0.026001,-0.036137,-0.014936,0.010191,-0.054925,0.002218,-0.018796,0.008872,-0.020398,0.012634,-0.01238,-0.007968,-0.012312,-0.020263,-0.00209,-0.009243,-0.040833
2025-09-01 00:00:00,-0.079514,-0.007902,0.01674,-0.084194,0.041544,-0.006755,0.05214,0.02049,0.011134,-0.045632,-0.017912,-0.022503,0.010938,-0.036934,0.000569,-0.018487,-0.013433,0.14513,-0.000585,-0.037668,-0.073803,0.002918,0.002412,-0.070007,-0.04946,-0.012605,0.001755,-0.070732,-0.013501
2025-10-01 00:00:00,-0.035847,-0.003582,-0.000671,-0.010772,0.002667,-0.020002,0.012929,0.000574,-0.008581,0.008332,-0.005884,0.015285,0.035611,0.01617,-0.017427,-0.026533,-0.008565,0.071521,0.004048,0.009318,-0.031003,-0.000894,0.032425,-0.03912,-0.024744,0.001404,0.002142,-0.037702,0.003393
2025-11-01 00:00:00,-0.001825,-0.104719,0.015314,-0.007322,0.014304,-0.008976,-0.014109,0.015831,0.033962,0.004241,-0.011459,-0.002418,0.018478,0.029768,-0.033993,0.012186,0.015636,-0.048898,0.005229,0.021251,0.004125,-0.029453,0.023208,0.017522,0.012415,0.002984,-0.002513,0.016298,0.012478


## 5. Data Profile

In [14]:
from utils import ML4T_DATA_PATH

# Check for profiles
for provider, subdir in [("Fama-French", "fama-french"), ("AQR", "aqr")]:
    profile_path = ML4T_DATA_PATH / "factors" / subdir / "profile.json"
    if profile_path.exists():
        profile = json.loads(profile_path.read_text())
        print(f"=== {provider} Profile ===")
        print(f"Files: {len(profile.get('files', []))}")
    else:
        print(f"{provider} profile not found at {profile_path}")

Fama-French profile not found at data/factors/fama-french/profile.json
AQR profile not found at data/factors/aqr/profile.json


## 6. Loader Options

The loaders support filtering by frequency and date range:

In [15]:
# Daily frequency
ff_daily = load_ff_factors(frequency="daily")
print(f"FF daily: {ff_daily.shape}")

FF daily: (15770, 7)


In [16]:
# Date range
recent_ff = load_ff_factors(start_date="2020-01-01")
print(f"FF 2020+: {recent_ff.shape}")

FF 2020+: (74, 7)


## 7. Documentation

### Fama-French Factors

From Ken French's Data Library:

| Factor | Description |
|--------|-------------|
| Mkt-RF | Market excess return |
| SMB | Small Minus Big (size) |
| HML | High Minus Low (value) |
| RMW | Robust Minus Weak (profitability) |
| CMA | Conservative Minus Aggressive (investment) |
| Mom | Momentum (12-1 month return) |

[Ken French Data Library](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html)

### AQR Factors

Alternative factors from AQR Capital:

| Factor | Description |
|--------|-------------|
| QMJ | Quality Minus Junk (profitability, growth, safety) |
| BAB | Betting Against Beta (low-beta premium) |
| VME | Value Minus Everything (alternative value) |
| HML Devil | Value with industry adjustment |

[AQR Datasets](https://www.aqr.com/Insights/Datasets)

## 8. Updating Data

To update with the latest data:

```python
# Update Fama-French factors
download_ff_factors()

# Update AQR factors
download_aqr_factors()
```

Factor data is typically updated monthly.

## Summary

| Item | Value |
|------|-------|
| Providers | Ken French, AQR |
| Frequencies | Monthly, Daily |
| Coverage | 1926-present (FF), varies (AQR) |
| API Key | None (free) |
| Loaders | `load_ff_factors()`, `load_aqr_factors()` |

**Primary use**: Risk attribution, alpha measurement, factor investing research.